In [ ]:
# Core
import pandas as pd
import numpy as np
import time

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
# My Functions
from static_training import train_static
from progressive_training import train_progressive
from mistake_training import train_mistake_driven


In [ ]:
df = pd.read_csv("data/electricity_market_dataset.csv")
# Convert timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort by time (VERY IMPORTANT)
df = df.sort_values('Timestamp').reset_index(drop=True)

df.head()
print("Shape:", df.shape)


Format the data

In [ ]:
target = "Electricity_Price_Forecast"
df_model = df.copy()

# Encode categorical columns
df_model["Regulatory_Policies"] = df_model["Regulatory_Policies"].map({"Low": 0, "Medium": 1, "High": 2})
df_model["Energy_Access_Data"] = df_model["Energy_Access_Data"].map({"Rural": 0, "Urban": 1})
df_model["Project_Risk_Analysis"] = df_model["Project_Risk_Analysis"].map({"Low": 0, "Medium": 1, "High": 2})


# Drop target and timestamps
X = df_model.drop(columns=[target, "Timestamp"])
y = df_model[target]

In [ ]:
# Plot Function
def makeRSMEPlot(df_results, vmin=.8, vmax=25, title="RMSE Heatmap"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_results.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Tree Depth")
    plt.ylabel("Look-Back (rows)")
    plt.show()
def makeTimePlot(df_time, vmin=0, vmax=9, title="Computation Time (s)"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_time.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Tree Depth")
    plt.ylabel("Look-Back (rows)")
    plt.show()

Define Static Function

Run fist experiment

In [ ]:
# Training window sizes (1, 3, 6, 12, 18, 24 months)


look_back_values = [720, 2160, 4320, 8640,  12960, 17280]  
tree_depths = [3, 5, 7]  

# Dictionaries to store RMSE and computation time
results_static = {}
time_results_static = {}

for lb in look_back_values:
    results_static[lb] = {}
    time_results_static[lb] = {}
    
    for depth in tree_depths:
        start_time = time.time()
        predictions, avg_rmse = train_static(X, y, look_back=lb, depthOfTree=depth)
        elapsed_time = time.time() - start_time

        # Store
        results_static[lb][depth] = avg_rmse
        time_results_static[lb][depth] = elapsed_time

        print(f"Look-back {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_static_rmse = pd.DataFrame(results_static).T 
df_static_time = pd.DataFrame(time_results_static).T
makeRSMEPlot(df_results=df_static_rmse, vmax= 28, vmin=20)
makeTimePlot(df_static_time, vmax=.5)

We will use the function to look ahead by 1 month, retraining every 3 months. Using the same training lengths and decision tree sizes

In [ ]:
retrain_step = 24*90 
look_ahead = 24 *30
results_progressive = {}
time_results_progressive = {}

for lb in look_back_values:
    results_progressive[lb] = {}
    time_results_progressive[lb] = {}
    for depth in tree_depths:
        start_time = time.time()
        _, avg_rmse = train_progressive(X, y, look_back=lb, look_ahead=look_ahead,
                                        retrain_step=24*90, depthOfTree=depth)
        elapsed_time = time.time() - start_time
        time_results_progressive[lb][depth] = elapsed_time
        results_progressive[lb][depth] = avg_rmse
        print(f"Look-back {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")


In [ ]:
df_progressive_rmse = pd.DataFrame(results_progressive).T 
makeRSMEPlot(df_progressive_rmse, vmin=.8, vmax=1.8)

df_progressive_time = pd.DataFrame(time_results_progressive).T
makeTimePlot(df_progressive_time, vmin=.2, vmax=7)

In [ ]:
threshold = 1.5           # retrain if RMSE > 3
results_mistake = {}
time_results_mistake = {}

for lb in look_back_values:
    results_mistake[lb] = {}
    time_results_mistake[lb] = {}
    for depth in tree_depths:
        start_time = time.time()
        _, avg_rmse = train_mistake_driven(X, y, look_back=lb, look_ahead=look_ahead,
                                           depthOfTree=depth, threshold=threshold)
        elapsed_time = time.time() - start_time
        
        results_mistake[lb][depth] = avg_rmse
        time_results_mistake[lb][depth] = elapsed_time
        
        print(f"Look-back {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_mistake_rmse = pd.DataFrame(results_mistake).T  # RMSE
df_mistake_time = pd.DataFrame(time_results_mistake).T  # Computation time
makeRSMEPlot(df_mistake_rmse)
makeTimePlot(df_mistake_time)

In [ ]:
makeRSMEPlot(df_static_rmse, title="Static RMSE")
makeRSMEPlot(df_progressive_rmse, title="Progressive RMSE")
makeRSMEPlot(df_mistake_rmse, title="Mistake-Driven RMSE")

makeTimePlot(df_static_time, title="Static Time (s)")
makeTimePlot(df_progressive_time, title="Progressive Time (s)")
makeTimePlot(df_mistake_time, title="Mistake-Driven Time (s)")


# Conclusion
For the electrical project the time savings seem to have an inverse relationship with the RSME value that comes back. If the retrain is set to a competitive level with progressive window it is possible to get RSME within striking distance struggles to keep things going